# 10. 의사결정나무 Study 데이터 실습 (Decision Tree Study)

`Study.공유.csv` 데이터를 전처리하고 `DecisionTreeClassifier`로 `survived` 예측 모델과 피처 중요도를 확인합니다.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, confusion_matrix

data_dir = DATA_DIR if 'DATA_DIR' in globals() else Path('data')
if not (data_dir / 'Study.공유.csv').is_file():
    data_dir = Path('Study_Python') / '[7] Machine_Learning' / 'data'
if not (data_dir / 'Study.공유.csv').is_file():
    data_dir = Path('..') / 'Study_Python' / '[7] Machine_Learning' / 'data'
data = pd.read_csv(data_dir / 'Study.공유.csv')

## 문제 1. 타깃 정제, 계층적 데이터 분할 및 누수 방지 결측 대체

`survived` 열의 결측 행을 제거하고, `ID`와 `survived`를 제외한 피처 `X`를 `stratify=y`, `test_size=0.2`, `random_state=42`로 먼저 분할한 뒤 훈련 세트(`X_tr`)의 평균으로 `X_tr`과 `X_te`의 결측치를 대체하고 클래스 분포를 확인하세요.

In [ ]:
target_col = 'survived'
df_clean = data.dropna(subset=[target_col]).copy()
feature_cols = [c for c in df_clean.columns if c not in ['ID', target_col]]
X = df_clean[feature_cols].copy()
y = df_clean[target_col].astype(int)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
train_means = X_tr.mean(numeric_only=True)
X_tr = X_tr.fillna(train_means)
X_te = X_te.fillna(train_means)
print('Train shape:', X_tr.shape, 'Test shape:', X_te.shape)
print('Target ratio:\n', y.value_counts(normalize=True).round(4))

**결과 해석**

`stratify=y`로 훈련/테스트 클래스 비율을 유지하고, 분할 이후 `X_tr`의 평균으로만 결측치를 대체하여 테스트 데이터 정보가 훈련에 유입되는 누수를 방지했습니다.

## 문제 2. 의사결정나무 학습, 혼동 행렬 및 피처 중요도 확인

`DecisionTreeClassifier(max_depth=4, min_samples_leaf=2, random_state=42)`를 학습시켜 `confusion_matrix`, `classification_report`, 그리고 상위 5개 피처 중요도를 출력하세요.

In [ ]:
target_col = 'survived'
df_clean = data.dropna(subset=[target_col]).copy()
feature_cols = [c for c in df_clean.columns if c not in ['ID', target_col]]
X = df_clean[feature_cols].copy()
y = df_clean[target_col].astype(int)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
train_means = X_tr.mean(numeric_only=True)
X_tr = X_tr.fillna(train_means)
X_te = X_te.fillna(train_means)

dt = DecisionTreeClassifier(max_depth=4, min_samples_leaf=2, random_state=42)
dt.fit(X_tr, y_tr)
y_pred = dt.predict(X_te)
print(confusion_matrix(y_te, y_pred))
print(classification_report(y_te, y_pred, digits=4))
imp = pd.Series(dt.feature_importances_, index=feature_cols).sort_values(ascending=False)
print('Top 5 Features:\n', imp.head(5))

**결과 해석**

`max_depth=4`와 `min_samples_leaf=2`로 과적합을 제어한 상태에서 모델의 분류 성능과 예측에 기여도가 큰 핵심 변수를 함께 파악할 수 있습니다.